# Lab L3.1 · Credit default classifier — logistic regression, k-NN, calibration and a cost-optimal cut-off
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 3 · ⏱ 80 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Turn the Day 1 credit contract into a decision-ready PD model, compare it with k-NN, set the cut-off from loss economics and add a multi-class risk grade.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the hint in the HTML lab guide, or peek at the Level 3 solution.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The Retail Credit Committee meets on Thursday. They need: a PD model better than the L1.2 baseline, evidence that its probabilities are **calibrated** (they feed IFRS 9 provisions), a recommended **approval cut-off** with its AED impact, and a five-grade risk scale for pricing.

**Progressive architecture:** consumes L1.2’s artefacts; its saved policy is registered on Day 5.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load the credit contract

**Why this matters.** Reusing the *fitted* artefacts guarantees the features are exactly what Model Risk reviewed on Day 1. We clone the preprocessor whenever we train a new model so each model owns an identical, independently fitted copy.

In [ ]:
import joblib, json
from sklearn.base import clone
feat = pd.read_parquet(CKPT / "loan_features.parquet")
spec = json.loads((CKPT / "loan_feature_spec.json").read_text())
preprocessor = joblib.load(CKPT / "loan_preprocessor.joblib")      # fitted on train in Lab L1.2
TARGET = spec["target"]
train, valid, test = (feat[feat.split == s] for s in ["train", "valid", "test"])
print({s: len(d) for s, d in zip(["train", "valid", "test"], [train, valid, test])}, "| honest baseline AUC from L1.2:", spec["honest_valid_auc"])

**✔ Checkpoint.** ≈ 208k / 46k / 46k rows; honest baseline AUC ≈ 0.83.

## Step 2 · Logistic regression and odds ratios

**Why this matters.** A credit committee wants to know *why* the model scores someone high. Logistic coefficients are log-odds; **exp(coef)** is the **odds ratio** per one-standard-deviation change of a scaled feature — directly explainable and the basis of adverse-action reasons.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — area under precision–recall
- **Blank 2** *(inline)* — odds ratio = e^coefficient

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
y_tr, y_va = train[TARGET].to_numpy(), valid[TARGET].to_numpy()

def scores(y, p) -> dict:
    return {"ROC_AUC": roc_auc_score(y, p), "PR_AUC": ____, "Brier": brier_score_loss(y, p)}  # ✏️ B1

logit = Pipeline([("prep", clone(preprocessor)), ("clf", LogisticRegression(max_iter=2000))]).fit(train, y_tr)
p_logit = logit.predict_proba(valid)[:, 1]
board = {"logistic": scores(y_va, p_logit)}
odds = pd.Series(____, index=logit.named_steps["prep"].get_feature_names_out())  # ✏️ B2
print("raise default odds:\n", odds.sort_values(ascending=False).head(8).round(2))
print("lower default odds:\n", odds.sort_values().head(5).round(2))
pd.DataFrame(board).T.round(4)

**✔ Checkpoint.** ROC-AUC ≈ 0.83, PR-AUC ≈ 0.33. DBR (≈ 1.6×), prior delinquencies (≈ 1.5×), weaker employer category and missing bureau file raise the odds; a higher bureau score (≈ 0.45×) and Government employment lower them.

## Step 3 · k-NN: the scaling experiment and tuning k

**Why this matters.** k-NN classifies by majority vote of the nearest neighbours — distance on raw AED salaries swamps every other feature. Scaling is not optional. k-NN is also a **lazy learner**: no training cost, but prediction cost grows with data, so we use samples.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — equalise feature scales

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
tr_s = train.sample(40_000, random_state=RNG); va_s = valid.sample(10_000, random_state=RNG)
RAW = ["monthly_salary_aed", "loan_amount_aed", "aecb_score", "debt_burden_ratio", "num_prior_delinquencies_24m", "num_bureau_inquiries_6m"]
for name, steps in [("unscaled", [SimpleImputer(strategy="median")]), ("scaled", [SimpleImputer(strategy="median"), ____])]:  # ✏️ B3
    knn = Pipeline([(f"s{i}", s) for i, s in enumerate(steps)] + [("knn", KNeighborsClassifier(n_neighbors=51))]).fit(tr_s[RAW], tr_s[TARGET])
    print(f"k-NN {name:8s} AUC = {roc_auc_score(va_s[TARGET], knn.predict_proba(va_s[RAW])[:, 1]):.3f}")

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — number of neighbours that vote

In [ ]:
ks, aucs = [5, 15, 51, 101, 201], []
for k in ks:
    m = Pipeline([("prep", clone(preprocessor)), ("knn", KNeighborsClassifier(n_neighbors=____))]).fit(tr_s, tr_s[TARGET])  # ✏️ B4
    aucs.append(roc_auc_score(va_s[TARGET], m.predict_proba(va_s)[:, 1]))
plt.plot(ks, aucs, "o-"); plt.xscale("log"); plt.xlabel("k"); plt.ylabel("validation AUC"); plt.title("k-NN: small k overfits, large k smooths"); plt.show()
best_k = ks[int(np.argmax(aucs))]
knn = Pipeline([("prep", clone(preprocessor)), ("knn", KNeighborsClassifier(n_neighbors=best_k))]).fit(tr_s, tr_s[TARGET])
t0 = time.perf_counter(); p_knn = knn.predict_proba(valid)[:, 1]; dt = time.perf_counter() - t0
board[f"kNN k={best_k} (40k sample)"] = scores(y_va, p_knn)
print(f"best k={best_k}; scoring {len(valid):,} loans took {dt:.1f}s"); pd.DataFrame(board).T.round(4)

**✔ Checkpoint.** Unscaled ≈ 0.60–0.65 vs scaled ≈ 0.75+. AUC rises with k up to ~100–200. k-NN trails logistic and is much slower to score.

## Step 4 · Imbalance: class weights vs calibration

**Why this matters.** With ~7% defaulters the model can be lazy about the minority class. `class_weight='balanced'` forces attention to defaulters — but distorts the probabilities. A **reliability (calibration) curve** shows whether a predicted 10% PD really defaults 10% of the time.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — weights inversely proportional to class frequency
- **Blank 6** *(inline)* — observed default rate vs mean predicted PD per bin

In [ ]:
from sklearn.calibration import calibration_curve
logit_bal = Pipeline([("prep", clone(preprocessor)), ("clf", LogisticRegression(max_iter=2000, class_weight=____))]).fit(train, y_tr)  # ✏️ B5
p_bal = logit_bal.predict_proba(valid)[:, 1]
board["logistic balanced"] = scores(y_va, p_bal)
for name, pp in [("logistic", p_logit), ("balanced", p_bal)]:
    frac_pos, mean_pred = ____  # ✏️ B6
    plt.plot(mean_pred, frac_pos, "o-", label=name)
plt.plot([0, .6], [0, .6], "k--", label="perfect"); plt.xlabel("mean predicted PD"); plt.ylabel("observed default rate"); plt.legend(); plt.show()
pd.DataFrame(board).T.round(4)

**✔ Checkpoint.** Same ROC-AUC; the unweighted model hugs the diagonal (low Brier) while the balanced model sits far below it — over-predicting PD.

## Step 5 · Cost-optimal cut-off with an approval-rate constraint

**Why this matters.** Credit policy is a business decision. Using **per-loan** economics — a missed default costs its actual loss; a wrongly rejected good loan costs forgone margin — we find the cost-minimising PD cut-off, then check it against the commercial constraint of ≥ 80% approvals.

#### ✏️ What to write in the next cell
- **Blank 7** *(code block, ≈1 lines)* — total = sum of loss_amount_aed over fn  +  sum of margin over fp
- **Blank 8** *(inline)* — commercial minimum approval rate

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
def policy_cost(p, d, t) -> float:
    reject = p >= t
    fn = (~reject) & (d[TARGET].to_numpy() == 1)              # approved defaulter
    fp = reject & (d[TARGET].to_numpy() == 0)                  # rejected good borrower
    margin = 0.03 * d.loan_amount_aed.to_numpy() * np.maximum(d.tenure_months.to_numpy(), 12) / 12
    # ✏️ BLANK 7 — write ≈1 line(s): total = sum of loss_amount_aed over fn  +  sum of margin over fp
    # ...your code here...

ths = np.round(np.arange(0.01, 0.51, 0.01), 2)
curve = pd.DataFrame({"t": ths, "cost": [policy_cost(p_logit, valid, t) for t in ths], "approval": [(p_logit < t).mean() for t in ths]})
t_cost = curve.loc[curve.cost.idxmin(), "t"]
t_policy = curve[curve.approval >= ____].sort_values("cost").iloc[0].t  # ✏️ B8
fig, ax = plt.subplots(); ax.plot(curve.t, curve.cost / 1e6, label="cost (AED m)"); ax2 = ax.twinx(); ax2.plot(curve.t, curve.approval, "g--", label="approval rate")
ax.axvline(t_cost, color="r", ls=":"); ax.axvline(t_policy, color="purple", ls="-."); ax.set_xlabel("PD cut-off"); ax.set_ylabel("AED m"); ax2.set_ylabel("approval"); plt.show()
for name, t in [("cost-optimal", t_cost), ("policy (≥80% approve)", t_policy), ("naive 0.5", 0.5)]:
    yhat = (p_logit >= t).astype(int)
    print(f"{name:22s} t={t:.2f} approve={(1 - yhat.mean()):.1%} cost=AED {policy_cost(p_logit, valid, t) / 1e6:6.1f}m "
          f"precision={precision_score(y_va, yhat):.2f} recall={recall_score(y_va, yhat):.2f} F1={f1_score(y_va, yhat):.2f}")

**✔ Checkpoint.** Cost-optimal t ≈ 0.24 with ≈ 92% approvals, so the ≥ 80% constraint is *not binding* here (both rows identical). The naive 0.5 cut-off costs ≈ AED 15m more and catches only ≈ 12% of defaulters vs ≈ 39%.

## Step 6 · Multi-class risk grade (A–E)

**Why this matters.** Pricing uses **grades**, not a binary flag. A multinomial logistic model predicts five ordered classes. With imbalanced classes, **macro-F1** (every grade equal) and **weighted-F1** (by frequency) tell different stories.

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — multi-class target A..E

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
tr_g = train.sample(80_000, random_state=RNG)
grader = Pipeline([("prep", clone(preprocessor)), ("clf", LogisticRegression(max_iter=3000))]).fit(tr_g, ____)  # ✏️ B9
g_pred = grader.predict(valid)
print(classification_report(valid.risk_grade, g_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(valid.risk_grade, g_pred, normalize="true", cmap="Purples", values_format=".2f"); plt.show()
dist = np.abs(valid.risk_grade.map("ABCDE".index).to_numpy() - pd.Series(g_pred).map("ABCDE".index).to_numpy())
print(f"exact: {(dist == 0).mean():.1%} | within one grade: {(dist <= 1).mean():.1%}")

**✔ Checkpoint.** Accuracy ≈ 67% (macro-F1 ≈ 0.59 < weighted-F1 ≈ 0.67), but ≈ 97% of predictions are within one grade; middle grades C–D are hardest.

## Step 7 · Out-of-time test and policy hand-off

**Why this matters.** All choices so far used validation data. The **test** split (mid-2024 → 2025) is touched **once** to estimate real-world performance. The policy JSON is the artefact the Credit Committee approves and Day 5 registers.

In [ ]:
p_test = logit.predict_proba(test)[:, 1]
oot = scores(test[TARGET], p_test)
policy = {"model": "logistic_regression_unweighted", "pd_cutoff": float(t_policy), "unconstrained_cost_optimal_cutoff": float(t_cost),
          "oot_test": {k: round(float(v), 4) for k, v in oot.items()}, "oot_approval_rate": float((p_test < t_policy).mean()),
          "oot_cost_aed_m": round(policy_cost(p_test, test, t_policy) / 1e6, 2), "knn_best_k": int(best_k)}
joblib.dump(logit, CKPT / "credit_model.joblib")
(CKPT / "credit_policy.json").write_text(json.dumps(policy, indent=2)); print(json.dumps(policy, indent=2))

**✔ Checkpoint.** OOT ROC-AUC within ≈ 0.01 of validation — the model generalises over time.

## ✅ Lab complete
The PD model, its calibration evidence and its cut-off are ready for the Credit Committee. **Next:** unsupervised learning — customer segments.